# Credit Card Fraud Detection — Complete Model Comparison

This notebook combines four distinct machine learning paradigms to tackle highly imbalanced fraud detection:
1. **Bagging (Random Forest)**
2. **Boosting (XGBoost)**
3. **Anomaly Detection (Isolation Forest)**
4. **Deep Learning (PyTorch DNN)**

It also integrates production-ready business logic including **SMOTE pipelines**, **Cost-based Threshold Selection**, and a **Fraud Review Queue**.

## Table of Contents
1. [Imports and Setup](#imports)
2. [Data Preprocessing](#prep)
3. [Supervised Bagging: SMOTE & Random Forest](#smote)
4. [Supervised Boosting: XGBoost](#xgb)
5. [Unsupervised Anomaly Detection: Isolation Forest](#iforest)
6. [Deep Learning: PyTorch DNN](#dnn)
7. [Cost-Based Threshold Optimization](#cost)
8. [Fraud Review Queue (Business Logic)](#queue)


## 1. Imports and Setup <a name="imports"></a>


In [1]:
import sys
import os
# Allow importing from the src directory in the parent folder
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier, IsolationForest
import xgboost as xgb

from sklearn.metrics import confusion_matrix, f1_score, classification_report
import matplotlib.pyplot as plt

# Imblearn Pipeline prevents data leakage during SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# Import advanced business logic from src
from src.review_queue import DecisionBands, create_review_queue, print_review_queue_summary

np.random.seed(2)
torch.manual_seed(2)


## 2. Data Preprocessing <a name="prep"></a>


In [2]:
df = pd.read_csv('../data/raw/creditcard.csv')

scaler = StandardScaler()
df['normalizedAmount'] = scaler.fit_transform(df['Amount'].values.reshape(-1, 1))

# Kept Amount for the review queue later
X_full = df.drop(['Time', 'Class'], axis=1)
y = df['Class']

X_train_full, X_test_full, y_train, y_test = train_test_split(X_full, y, test_size=0.3, random_state=0, stratify=y)

# Datasets for modeling (dropping the raw Amount column)
X_train = X_train_full.drop(['Amount'], axis=1).values
X_test = X_test_full.drop(['Amount'], axis=1).values
y_train = y_train.values.ravel()
y_test = y_test.values.ravel()

# class imbalance ratio for XGBoost
neg_class_count = (y_train == 0).sum()
pos_class_count = (y_train == 1).sum()

imbalance_ratio = neg_class_count / pos_class_count

print(f"Class Imbalance Ratio (Neg/Pos): {imbalance_ratio:.2f}")
print(f"means we have 1 positive sample (fraud) for every {imbalance_ratio:.2f} negative samples (non-fraud) .")


Class Imbalance Ratio (Neg/Pos): 578.55
means we have 1 positive sample (fraud) for every 578.55 negative samples (non-fraud) .


## 3. Supervised Bagging : SMOTE & Random Forest
Using an ImbPipeline to ensures that SMOTE (Synthetic Minority Over-sampling Technique) is only applied to training data this prevents data leakage .


In [3]:
smote_rf_pipeline = ImbPipeline([
    ("smote", SMOTE(random_state = 2)),
    ("model", RandomForestClassifier(n_estimators = 100, random_state = 2, n_jobs = -1))
])

print("Training SMOTE + Random Forest |||||")
smote_rf_pipeline.fit(X_train, y_train)
rf_probs = smote_rf_pipeline.predict_proba(X_test)[:, 1]
print(f"RF Default F1 score is (0.5 threshold): {f1_score(y_test, rf_probs > 0.5):.4f}")


Training SMOTE + Random Forest |||||
RF Default F1 score is (0.5 threshold): 0.8406


## 4. Supervised Boosting: XGBoost

XGBoost build trees sequentially to correct its previous errors. By setting `scale_pos_weight` to the imbalance ratio , we mathematically force the model to heavily penalize missing a fraud case.

In [4]:
print("Training XGBoost |||||")
xgb_model = xgb.XGBClassifier(
    scale_pos_weight = imbalance_ratio,
    random_state = 2,
    n_jobs = -1,
    eval_metric = 'aucpr'   # becoz we are dealing with imbalanced data, AUC-PR is more informative than AUC-ROC
)

xgb_model.fit(X_train, y_train)
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]

print(f"XGBoost Default F1 score is (0.5 threshold): {f1_score(y_test, xgb_probs > 0.5):.4f}")

Training XGBoost |||||
XGBoost Default F1 score is (0.5 threshold): 0.8487


## 5. Unsupervised Anomaly Detection: Isolation Forest <a name="iforest"></a>
Instead of finding a decision boundary, Isolation Forest isolates anomalies. Fraud is inherently anomalous, making this unsupervised approach highly effective without needing SMOTE.


In [5]:
print("Training Isolation Forest |||||")

# Contamination is the expected proportion of outliers (fraud rate)
fraud_rate = len(df[df['Class'] == 1]) / len(df)    # number of fraud cases / total number of cases
iso_forest = IsolationForest(contamination=fraud_rate, random_state=2, n_jobs=-1)

# Unsupervised tech. so We only fit on X_train. It doesn't use y_train !
iso_forest.fit(X_train)

# Returns 1 for normal, -1 for anomaly. We map -1 to 1 (Fraud) and 1 to 0 (Normal)
iso_preds = iso_forest.predict(X_test)
iso_preds = np.where(iso_preds == -1, 1, 0)

print(f"Isolation Forest F1 Score: {f1_score(y_test, iso_preds):.4f}")


Training Isolation Forest |||||
Isolation Forest F1 Score: 0.2215


## 6. Deep Learning: PyTorch DNN <a name="dnn"></a>


In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class FraudDetection(nn.Module):

    def __init__(self, input_dim=29):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 16), nn.ReLU(),
            nn.Linear(16, 24), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(24, 20), nn.ReLU(),
            nn.Linear(20, 24), nn.ReLU(),
            nn.Linear(24, 1), nn.Sigmoid()
        )


    def forward(self, x):
        return self.net(x)

model = FraudDetection().to(device)
criterion = nn.BCELoss()        # Binary Cross Entropy Loss for binary classification => it penalizes the model more for being confident and wrong, which is what we want in fraud detection.
optimizer = optim.Adam(model.parameters())

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=15, shuffle=True)

print("Training DNN |||||")

for epoch in range(20):
    model.train()
    for bx, by in train_loader:                 # fetch a batch of data
        bx, by = bx.to(device), by.to(device)   # move data to GPU if available
        optimizer.zero_grad()      
        loss = criterion(model(bx), by)
        loss.backward()
        optimizer.step()                        # update neural network weights

model.eval()                                    # no more training , we are in evaluation mode so Dropout is disabled during evaluation
with torch.no_grad():                           
    dnn_probs = model(X_test_t).cpu().numpy().flatten()
    
print(f"DNN Default F1 (0.5 threshold): {f1_score(y_test, dnn_probs > 0.5):.4f}")


Training DNN |||||
DNN Default F1 (0.5 threshold): 0.7770


## 7. Cost-Based Threshold Optimization <a name="cost"></a>
In business, a False Negative (missed fraud) costs significantly more than a False Positive (false alarm). Here, we simulate minimizing total financial cost on our top-performing model (XGBoost).


In [7]:
def calculate_cost(y_true, y_prob, threshold, fn_cost=10, fp_cost=1):
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return (fn * fn_cost) + (fp * fp_cost)


thresholds = np.arange(0.1, 0.9, 0.05)

costs = [calculate_cost(y_test, xgb_probs, t) for t in thresholds]

best_idx = np.argmin(costs)
optimal_threshold = thresholds[best_idx]
print(f"Optimal Threshold for XGBoost (lowest cost): {optimal_threshold:.2f}")
print(f"Cost at 0.50: {calculate_cost(y_test, xgb_probs, 0.50)}")
print(f"Cost at {optimal_threshold:.2f}: {costs[best_idx]}")


Optimal Threshold for XGBoost (lowest cost): 0.10
Cost at 0.50: 338
Cost at 0.10: 314


## 8. Fraud Review Queue (Business Logic) <a name="queue"></a>
In reality, banks use **Decision Bands** to route uncertain transactions to human analysts. Let's apply our XGBoost probabilities to the Review Queue logic from our `src` folder.


In [8]:
decision_bands = DecisionBands(
    approve_threshold = 0.20, 
    block_threshold = optimal_threshold
)

# Create the review queue dataframe using the raw test data (which contains original Amount)
review_queue = create_review_queue(
    X_test=X_test_full,
    y_test=pd.Series(y_test),
    y_prob=xgb_probs,
    decision_bands=decision_bands,
    feature_cols=['Amount']
)

print_review_queue_summary(review_queue)

ValueError: Invalid thresholds: 0 <= approve(0.2) < block(0.1) <= 1

## Conclusion

This notebook demonstrates a comprehensive approach to credit card fraud detection, leveraging multiple machine learning techniques and integrating business logic for practical application. By comparing the performance of different models and optimizing for cost, we can effectively identify fraudulent transactions while minimizing financial loss.